In [105]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from utils.helpers import *
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

device = find_backend()

Currently using:  mps


In [106]:
from transformers import AutoTokenizer, DistilBertModel
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# teacher_model = DistilBertModel.from_pretrained("distilbert-base-uncased")

In [107]:
from models import decoder_only

# model_type = "encoder_decoder"
model_type = "decoder_only"

learning_rate = 0.0003
batch_size = 5
epochs = 10


vocab_size = tokenizer.vocab_size
hidden_dim=128
num_heads=2
dim_feedforward=2048
num_layers_dec=2
dropout=0.1
max_length=128
p_tags=5
ignore_index = tokenizer.pad_token_id

model = decoder_only.DecoderModel(vocab_size, device, hidden_dim, num_heads, dim_feedforward, num_layers_dec, dropout, max_length, p_tags, ignore_index = 0, sos_index=tokenizer.cls_token_id)


In [ ]:
import requests
text = requests.get()
for line in data: # files are iterable
    print line

In [1]:
questions

NameError: name 'questions' is not defined

In [7]:
if not a:
    print('empty list')

In [9]:
import torch

In [18]:
next_token_logits = torch.rand((2,4))

In [70]:
next_token_logits

tensor([[0.5795, 0.5144, 0.9737, 0.5004],
        [0.8829, 0.0982, 0.3406, 0.7243]])

In [ ]:
sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True, dim=-1)
sorted_logits = F.softmax(sorted_logits, dim = -1)
cum_prob = torch.cumsum(sorted_logits, dim = -1)

In [101]:
cum_prob

tensor([[0.3414, 0.5716, 0.7873, 1.0000],
        [0.3459, 0.6411, 0.8422, 1.0000]])

In [73]:
sorted_indices

tensor([[2, 0, 1, 3],
        [0, 3, 2, 1]])

In [82]:
next_token_logits

tensor([[0.5795, 0.5144, 0.9737, 0.5004],
        [0.8829, 0.0982, 0.3406, 0.7243]])

In [91]:
remove = cum_prob > 0.85
sorted_logits[remove] = float("-inf")

In [94]:
full_array = torch.ones_like(next_token_logits)
full_array.scatter_(1, sorted_indices, sorted_logits)
F.softmax(full_array, dim=-1)

tensor([[0.3222, 0.3176, 0.3602, 0.0000],
        [0.3551, 0.0000, 0.3073, 0.3376]])

In [95]:
top_p = 0.85
filtered_logits = next_token_logits.clone()

In [103]:
# Apply nucleus (top-p) sampling
if top_p < 1.0:
    # Sort logits in descending order
    sorted_logits, sorted_indices = torch.sort(filtered_logits, descending=True, dim=-1)
    print(sorted_logits)
    
    # Calculate cumulative probabilities
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    print(cumulative_probs)
    
    # Find indices to remove
    sorted_indices_to_remove = cumulative_probs > top_p
    print(sorted_indices_to_remove)
    # Keep the first token above threshold
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0
    
    # Apply -inf to filtered positions in sorted space
    sorted_logits[sorted_indices_to_remove] = float('-inf')
    print(sorted_logits)
    
    # Return to original indexing
    # Create empty logits tensor
    filtered_logits = torch.full_like(filtered_logits, float('-inf'))
    
    # For each item in batch (usually just 1)
    batch_size = filtered_logits.shape[0]
    vocab_size = filtered_logits.shape[-1]
    
    # Create batch indices tensor
    batch_indices = torch.arange(batch_size, device=filtered_logits.device).view(-1, 1).expand(-1, vocab_size)
    
    # Scatter operation to put values back in original positions
    filtered_logits.scatter_(1, sorted_indices, sorted_logits)

tensor([[0.9737, 0.5795, 0.5144, 0.5004],
        [0.8829, 0.7243, 0.3406, 0.0982]])
tensor([[0.3414, 0.5716, 0.7873, 1.0000],
        [0.3459, 0.6411, 0.8422, 1.0000]])
tensor([[False, False, False,  True],
        [False, False, False,  True]])
tensor([[0.9737, 0.5795, 0.5144, 0.5004],
        [0.8829, 0.7243, 0.3406, 0.0982]])


In [100]:
tensor([[0.3414, 0.5716, 0.7873, 1.0000],
        [0.3459, 0.6411, 0.8422, 1.0000]])

tensor([[0.5795, 0.5144, 0.9737, 0.5004],
        [0.8829, 0.0982, 0.3406, 0.7243]])

In [65]:
next_token_logits

tensor([[0.5795, 0.5144, 0.9737, 0.5004],
        [0.8829, 0.0982, 0.3406, 0.7243]])

In [104]:
tokenizer._eos_token

NameError: name 'tokenizer' is not defined